# Cardiovascular Disease Prediction

A student-style machine-learning project using the supplied cardiovascular dataset. The analysis includes data cleaning, exploratory plots, a feature correlation matrix, and five classifier comparisons. This model is for coursework only, not clinical diagnosis.

## 1. Load the data and libraries
The CSV uses semicolons between columns. Change `DATA_PATH` if the file is stored somewhere else.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = Path("/Users/chitrarasanmacbookair/Downloads/cardio_train (1).csv")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
df_raw = pd.read_csv(DATA_PATH, sep=";")
print("Rows and columns:", df_raw.shape)
df_raw.head()

## 2. Preprocessing
Drop the record ID, remove duplicates, convert age from days to years, filter obvious measurement errors, and calculate BMI. The target is `cardio`.

In [ ]:
df = df_raw.drop(columns=["id"], errors="ignore").drop_duplicates().copy()
df["age_years"] = (df["age"] / 365.25).round(1)
df = df.drop(columns=["age"])
valid = (df.ap_hi.between(70, 250) & df.ap_lo.between(40, 200) &
         (df.ap_hi >= df.ap_lo) & df.height.between(120, 220) &
         df.weight.between(30, 200))
df = df.loc[valid].reset_index(drop=True)
df["bmi"] = df.weight / (df.height / 100) ** 2
print("Rows after cleaning:", len(df), "| removed:", len(df_raw)-len(df))
print("Missing values:", int(df.isna().sum().sum()))
df.head()

## 3. Target and feature exploration
The target is nearly balanced. These figures compare disease groups across measurements and recorded lifestyle categories. Group differences are associations, not evidence of causation.

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2")
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x="cardio", ax=ax)
ax.set(title="Cardiovascular disease class balance", xlabel="Cardio (0 = no disease, 1 = disease)", ylabel="Number of people")
plt.tight_layout()
plt.show()

In [ ]:
for col in ["age_years", "height", "weight", "bmi", "ap_hi", "ap_lo"]:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(data=df, x=col, hue="cardio", stat="density", common_norm=False, kde=True, element="step", ax=ax)
    ax.set_title(f"{col.replace('_', ' ').title()} distribution by disease status")
    plt.tight_layout(); plt.show()

In [ ]:
for col in ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]:
    rates = df.groupby(col, observed=True).cardio.mean().reset_index()
    fig, ax = plt.subplots(figsize=(6,4))
    sns.barplot(data=rates, x=col, y="cardio", ax=ax)
    ax.set(title=f"Disease proportion by {col}", ylabel="Proportion with disease", ylim=(0,1))
    plt.tight_layout(); plt.show()

## 4. Descriptive statistics and correlations

In [ ]:
display(df.describe().T)
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11,9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Feature correlation matrix")
plt.tight_layout(); plt.show()

## 5. Train and compare models
Use an 80:20 stratified train/test split with a fixed random seed. Median imputation and standardization are inside each pipeline, so these steps are learned from the training portion only. We compare SVM, KNN, Decision Tree, Logistic Regression, and Random Forest. Accuracy and ROC-AUC are shown; accuracy can be misleading by itself, so the ROC-AUC adds a second view of ranking performance.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

TARGET = "cardio"
FEATURES = [c for c in df.columns if c != TARGET]
X, y = df[FEATURES], df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)
prep = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), FEATURES)])
models = {"SVM": SVC(kernel="rbf", C=1, gamma="scale"), "KNN": KNeighborsClassifier(n_neighbors=9),
          "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=42),
          "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
          "Random Forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=42)}
results, fitted = [], {}
for name, classifier in models.items():
    pipe = Pipeline([("preprocess", prep), ("model", classifier)])
    pipe.fit(X_train, y_train)
    predicted = pipe.predict(X_test)
    scores = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, "predict_proba") else pipe.decision_function(X_test)
    cv = cross_val_score(pipe, X_train, y_train, cv=3, scoring="accuracy")
    results.append({"Model":name, "Test accuracy":accuracy_score(y_test,predicted), "ROC-AUC":roc_auc_score(y_test,scores), "3-fold CV accuracy":cv.mean()})
    fitted[name] = (pipe, predicted)
results = pd.DataFrame(results).sort_values("Test accuracy", ascending=False)
display(results)
fig, ax = plt.subplots(figsize=(8,4.5)); sns.barplot(data=results, x="Model", y="Test accuracy", ax=ax); ax.set_ylim(0,1); ax.tick_params(axis="x", rotation=20); ax.set_title("Test accuracy of five classifiers"); plt.tight_layout(); plt.show()

## 6. Select a model and inspect the errors
The model with the highest test accuracy is selected for this run. The confusion matrix and class report show how it performed on each class.

In [ ]:
best_name = results.iloc[0]["Model"]
best_model, best_pred = fitted[best_name]
print("Selected model:", best_name)
print(classification_report(y_test, best_pred, target_names=["No disease", "Disease"]))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=["No disease", "Disease"], cmap="Blues")
plt.title(f"Confusion matrix: {best_name}"); plt.tight_layout(); plt.show()
joblib.dump(best_model, OUT / "cardiovascular_model.joblib")
print("Saved model to", OUT / "cardiovascular_model.joblib")

## 7. Conclusion
This workflow compares common classifiers and includes preprocessing, visual exploration, correlations, and a holdout evaluation. Performance here applies only to this dataset and split; external validation and medical review would be necessary before any real-world clinical use.